# 02 — Data Cleaning, the Rules and the Objective

---

### What this notebook does

Two jobs.

1. **Cleans and prepares** the raw pack into a small set of model-ready files.
2. **Writes down the model** — the objective function and the seven constraints from
   `DOM Equations.docx` — as `dom_model.py`.

### Why it is the only place the model appears

`03_baseline`, `04_greedy` and `05_classical` all start with `from dom_model import *`. They do
no cleaning, no merging and no re-deriving. That is what makes the comparison in `06` fair: the
three methods differ **only** in how they search, never in what they are searching over.

So the rules are explained here, once, and nowhere else.

### The model in one line

`Objective = Revenue − Penalty − Shipping cost`, maximised, subject to C1–C7 below.

| # | Rule |
|---|---|
| C1 | One order goes to one DC only |
| C2 | You cannot fill more than what was ordered |
| C3 | Stock must be there, and must still cover the next 5 days |
| C4 | Move only if fill rate rises 5 points **and** 100 cases |
| C5 | Case picks + pallet picks must fit the DC's daily limit |
| C6 | One dock slot per order, docks are limited |
| C7 | Penalty applies only if filled cases fall under the order's threshold |

## 1. Setup

### 1.1 Imports and the shared work folder

In [1]:
import math, shutil
import numpy as np                       # arrays, used for the daily stock timeline
import pandas as pd                      # tables
from collections import defaultdict      # dict that creates empty sets by itself

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

Since in notebook 01 we have already uploaded the zip and the files were uploaded to the drive, then no need to reupload the zip again. The needed files are taken from the drive.

In [2]:
import os, glob, zipfile

# Colab keeps nothing between runtimes, so the shared folder goes on Drive when we can.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/DOM"
except Exception:
    WORK = os.path.abspath("dom_work")

CLEAN   = f"{WORK}/clean"      # notebook 02 writes here
RESULTS = f"{WORK}/results"    # notebooks 03, 04, 05 write here
os.makedirs(CLEAN, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
print("work folder:", WORK)

Mounted at /content/drive
work folder: /content/drive/MyDrive/DOM


### 1.2 Find the data

In [3]:
def find_dom_root():
    # search for a folder named DOM-data below the current folder, /content or WORK
    for pat in [f"{WORK}/**/DOM-data", "/content/**/DOM-data", "**/DOM-data"]:
        hits = [h for h in glob.glob(pat, recursive=True) if os.path.isdir(h)]
        if hits:
            return hits[0]
    return None

ROOT = os.environ.get("DOM_ROOT") or find_dom_root()   # is it already unzipped?
if ROOT is None:                                       # no -> we need the zip
    zips = glob.glob(f"{WORK}/*.zip") + glob.glob("/content/*.zip") + glob.glob("*.zip")
    if not zips:                                       # none there either
        from google.colab import files
        zips = list(files.upload().keys())             # ask the user to upload it
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(f"{WORK}/dom_extract")            # unzip it
    ROOT = find_dom_root()                             # look again

assert ROOT, "DOM-data folder not found"
IN = os.path.join(ROOT, "input data")                  # the 5 input files live here
print("found:", ROOT)

found: /content/drive/MyDrive/DOM/dom_extract/Nestle - WISER WQ26 DOM-data [SHARED]/DOM-data


### 1.3 Load the files

In [4]:
orders = pd.read_csv(f"{IN}/input_order data.csv")           # one row per order + SKU
cap    = pd.read_csv(f"{IN}/input_capacity_planning.csv")    # daily stock per DC and SKU
dock   = pd.read_csv(f"{IN}/input_dock_capacity.csv")        # daily dock slots per DC
ship   = pd.read_csv(f"{IN}/input_shipping_cost_data.csv")   # cost per DC -> zip lane
thru   = pd.read_csv(f"{IN}/input_throughput_capacity.csv")  # daily pick workload per DC

poc_ord = pd.read_csv(f"{ROOT}/Output_order_level_data.csv")      # check only
poc_sku = pd.read_csv(f"{ROOT}/output_order_sku_level_data.csv")  # check only

for n, d in [("orders",orders), ("capacity",cap), ("dock",dock),
             ("shipping",ship), ("throughput",thru)]:
    print(f"{n:11s} {d.shape}")                              # rows, columns

orders      (25193, 39)
capacity    (377504, 23)
dock        (480, 13)
shipping    (12922, 7)
throughput  (530, 7)


## 2. Clean the order book

### 2.1 Dates and flags

Parse the two dates the model needs and tidy the stock flag. The FTL and open-order rules are
already applied to this extract, which notebook `01` verified.

In [5]:
orders["PGI"] = pd.to_datetime(orders["transportationplanningdate"], format="%m/%d/%y")  # ship day
orders["RDD"] = pd.to_datetime(orders["RequestedDeliveryDate"],      format="%m/%d/%y")  # due day
orders["IsInvAvail"] = orders["IsInvAvail"].astype(str).str.strip().str.upper()          # tidy Y/N

print("orders:", orders.Group_Flag.nunique(), "| SKUs:", orders.MaterialNumber.nunique())

orders: 1109 | SKUs: 1110


### 2.2 Put every line into cases

The doc works in cases. Lines measured in pallets (`PL`) are converted with
`ProductCasesPerPallet`. Notebook `01` showed this reproduces the POC's `OrderedQty_Cases` on
100% of lines.

In [6]:
orders["cases"] = np.where(
    orders["ProductPlanningUnitOfMeasure"].eq("PL"),                     # is the line in pallets?
    orders["OrderedQty_converted"] * orders["ProductCasesPerPallet"],    # yes -> convert to cases
    orders["OrderedQty_converted"]).astype(float)                        # no  -> already cases

print("total cases ordered:", f"{orders.cases.sum():,.0f}")

total cases ordered: 2,554,440


### 2.3 Price per case and cases per pallet

Price per case lets us rebuild revenue from filled cases, so revenue always follows the fill.
Cases per pallet is needed for the pick split in C5.

In [7]:
orders["price_per_case"] = np.where(orders["cases"] > 0,                 # skip zero-case lines
                                    orders["Order_SKU_Revenue"] / orders["cases"], 0.0)
orders["cpp"] = (orders["ProductCasesPerPallet"]
                 .replace(0, np.nan)      # 0 would break the division
                 .fillna(1)               # missing -> treat as 1 case per pallet
                 .astype(float))
print(orders[["cases","price_per_case","cpp"]].describe().round(2).to_string())

          cases  price_per_case       cpp
count  25193.00        25193.00  25193.00
mean     101.39           43.71    157.42
std      235.41           58.96    102.67
min        1.00            5.44      1.00
25%       20.00           15.44     96.00
50%       48.00           28.70    128.00
75%      113.00           50.90    216.00
max     7850.00         3573.67    680.00


### 2.4 One row per order

Roll the lines up: default DC, dates, zip, priority, revenue, total cases and the two penalty
numbers. Both penalty fields are identical on every line of an order, so taking the first is safe.

In [8]:
head = orders.groupby("Group_Flag").agg(
    default_dc=("Plant","first"),          # the DC it is assigned to today
    pgi=("PGI","first"),                   # planned ship day
    rdd=("RDD","first"),                   # customer due day
    zipc=("ZipCode","first"),              # ship-to zip, used for the freight cost
    prio=("DeliveryPriority","min"),       # 8 = protected order, 99 = normal
    revenue=("Order_SKU_Revenue","sum"),   # total order value
    ordered_cases=("cases","sum"),         # total cases asked for
    frt=("FillRateThreshold","first"),     # fill % below which a penalty starts
    ppc=("Penaltyforpotentialcuts","first"),  # penalty rate on the missing value
).reset_index()

head[["frt","ppc"]] = head[["frt","ppc"]].fillna(0.0)          # no deal -> no penalty
head["threshold_cases"] = head["frt"] * head["ordered_cases"]  # turn the % into cases
print("orders:", len(head), "| no penalty schedule:", int((head.ppc == 0).sum()))

orders: 1109 | no penalty schedule: 370


### 2.5 The focus list — 472

Short on stock (any line `IsInvAvail = N`) gives 447. A fully booked dock day on the planned ship
date adds 25 more that were otherwise fine. Everything else is clean.

`is_focus` is stored on the order header, so the three solver notebooks all read the same list.

In [9]:
roll   = orders.groupby("Group_Flag")["IsInvAvail"].apply(lambda s: (s == "Y").all())  # all Y?
SUFF   = set(roll[roll].index)     # every line is fine
INSUFF = set(roll[~roll].index)    # at least one line is short

dock["Date"] = pd.to_datetime(dock["Date"], format="%m/%d/%y")        # tidy the date column
zero_days = set(map(tuple,                                            # set of (plant, day)
                    dock.loc[dock["Dock_Remaining"] == 0, ["Plant","Date"]]
                        .drop_duplicates().values))                   # days with no free dock

HEADD   = head.set_index("Group_Flag").to_dict("index")
FLAGGED = {gf for gf, h in HEADD.items()                               # does the order ship
           if (h["default_dc"], h["pgi"]) in zero_days}                # on such a day?
ADDED   = FLAGGED & SUFF                    # keep only ones that were otherwise fine
FOCUS   = sorted(INSUFF | ADDED)            # short ones + newly flagged ones

head["is_focus"] = head["Group_Flag"].isin(FOCUS)
print(f"{len(INSUFF)} short + {len(ADDED)} dock-blocked = {len(FOCUS)} FOCUS ORDERS")
print(f"clean orders: {int((~head.is_focus).sum())}")

447 short + 25 dock-blocked = 472 FOCUS ORDERS
clean orders: 637


## 3. Build the model inputs

### 3.1 A shared calendar

Everything that changes by day (stock, docks, picks) sits on the same day grid. Then "the next 5
days" is just a slice of an array.

In [10]:
cap["DATE"] = pd.to_datetime(cap["DATE"])
DMIN, DMAX  = cap["DATE"].min(), cap["DATE"].max()   # first and last day in the pack
DATES = pd.date_range(DMIN, DMAX, freq="D")          # every day in between
NT    = len(DATES)                                   # how many days
DIX   = {d: i for i, d in enumerate(DATES)}          # date  -> position in the array
DCS   = sorted(int(x) for x in orders["Plant"].unique())   # the 8 DCs
DCIX  = {d: i for i, d in enumerate(DCS)}            # DC    -> row number
print(f"{DMIN.date()} to {DMAX.date()} ({NT} days) | DCs: {DCS}")

2024-06-20 to 2024-07-21 (32 days) | DCs: [5083, 5385, 5410, 5420, 5490, 5620, 5641, 5773]


### 3.2 Stock

Free cases = `OpeningStock − Total_Reserved_Qty`. Each `(DC, SKU)` becomes one array over the
days, so taking stock on a day can also remove it from every later day — that is what makes the
5-day check in C3 mean something.

In [11]:
cap["AV"] = cap["OpeningStock"] - cap["Total_Reserved_Qty"]   # cases we may use
capw = cap[cap["LocationID"].isin(DCS)]                       # keep only our 8 DCs

POOL0 = {}
for (d, s), g in capw.groupby(["LocationID","MaterialID"], sort=False):
    arr = np.zeros(NT)                                    # one slot per day, start empty
    ix  = g["DATE"].map(DIX).dropna().astype(int).values   # which day each row belongs to
    arr[ix] = g["AV"].values[:len(ix)]                     # drop the stock into those days
    POOL0[(int(d), int(s))] = arr                          # save the timeline

print("(DC,SKU) series:", len(POOL0))

(DC,SKU) series: 11550


### 3.3 Shipping cost and lead time

The rate card is complete, so every DC can quote every ship-to zip. Lead time is not given but is
`ceil(distance / 500)`, verified against the POC in notebook `01`.

In [12]:
lanes = ship[["Plant","TargetZip","Distance","Shipping_Cost"]].drop_duplicates()
print("lanes:", len(lanes), "| DCs:", lanes.Plant.nunique(), "| zips:", lanes.TargetZip.nunique())

lanes: 12922 | DCs: 14 | zips: 923


### 3.4 Docks

`Dock_Remaining` is the number of free slots. Two DCs (5083, 5773) have no dock rows at all, so
we mark them "no data" and skip the check there. Treating them as zero would wrongly block every
move into them.

In [13]:
DOCK0 = np.full((len(DCS), NT), np.nan)          # start every DC/day as "unknown"
for r in dock.itertuples(index=False):
    if r.Plant in DCIX and r.Date in DIX:        # only DCs and days we track
        DOCK0[DCIX[r.Plant], DIX[r.Date]] = max(0.0, float(r.Dock_Remaining))

DOCK_HAS = ~np.isnan(DOCK0)                      # True where we really have data
DOCK0    = np.nan_to_num(DOCK0, nan=0.0)         # turn the unknowns into 0
print("days with dock data:", {d: int(DOCK_HAS[DCIX[d]].sum()) for d in DCS})

days with dock data: {5083: 0, 5385: 30, 5410: 30, 5420: 30, 5490: 30, 5620: 30, 5641: 30, 5773: 0}


### 3.5 Pick capacity

C5 needs a pick limit, but the file only reports usage, never a limit. So we take the limit to be
the **highest usage ever seen at that plant**, and what is left on a day is that peak minus the
usage already booked.

In [14]:
thru["transportationplanningdate"] = pd.to_datetime(thru["transportationplanningdate"])
CP_CAP = thru.groupby("Plant")["util_case_picks"].max().to_dict()   # peak case picks per DC
PP_CAP = thru.groupby("Plant")["util_pallets"].max().to_dict()      # peak pallet picks per DC

CP0 = np.zeros((len(DCS), NT))                    # case-pick room per DC/day
PP0 = np.zeros((len(DCS), NT))                    # pallet-pick room per DC/day
for d in DCS:
    CP0[DCIX[d], :] = CP_CAP.get(d, 0.0)          # start every day at full capacity
    PP0[DCIX[d], :] = PP_CAP.get(d, 0.0)
for r in thru.itertuples(index=False):            # then take off what is already used
    if r.Plant in DCIX and r.transportationplanningdate in DIX:
        i, j = DCIX[r.Plant], DIX[r.transportationplanningdate]
        CP0[i, j] = max(0.0, CP_CAP[r.Plant] - r.util_case_picks)
        PP0[i, j] = max(0.0, PP_CAP[r.Plant] - r.util_pallets)
print(pd.DataFrame({"case_pick_cap": CP_CAP, "pallet_pick_cap": PP_CAP}).to_string())

      case_pick_cap  pallet_pick_cap
5083          29325              287
5385         110295             1804
5410          66122             1238
5420          72659             1644
5490          57284              826
5620         133164             1426
5641           8838              409
5773           4634               47


## 4. Save the cleaned files

Five artefacts. Nothing else leaves this notebook.

| file | what it holds |
|---|---|
| `order_head.csv` | one row per order: DC, dates, zip, priority, revenue, cases, penalty terms, `is_focus` |
| `order_lines.csv` | one row per order line: SKU, cases, price per case, cases per pallet |
| `lanes.csv` | the freight rate card, DC → ship-to zip |
| `poc_reference.csv` | the 2024 POC's own answer, kept only for cross-checks |
| `model_grids.npz` | the calendar, the stock timelines, the dock grid and the pick grids |
| `dom_model.py` | the rules and the objective (written in section 5) |

In [15]:
head.to_csv(f"{CLEAN}/order_head.csv", index=False)
orders[["Group_Flag","MaterialNumber","cases","price_per_case","cpp"]] \
     .to_csv(f"{CLEAN}/order_lines.csv", index=False)
lanes.to_csv(f"{CLEAN}/lanes.csv", index=False)

poc_ord.rename(columns={"SalesDocument/GroupingIndicator": "order"})[
    ["order","IsDivert","DefaultDC","RecommendedDC","Default_DC_COF",
     "OrderedQty_Cases","Default_Qty_Cases"]].to_csv(f"{CLEAN}/poc_reference.csv", index=False)

pool_keys = np.array(list(POOL0.keys()), dtype=np.int64)         # (DC, SKU) pairs
pool_vals = np.vstack([POOL0[tuple(k)] for k in pool_keys])      # one stock row per pair
np.savez_compressed(f"{CLEAN}/model_grids.npz",
                    dates=DATES.values, dcs=np.array(DCS, dtype=np.int64),
                    pool_keys=pool_keys, pool_vals=pool_vals,
                    dock=DOCK0, dock_has=DOCK_HAS, cp=CP0, pp=PP0)

for f in ["order_head.csv","order_lines.csv","lanes.csv","poc_reference.csv","model_grids.npz"]:
    print(f"{f:20s} {os.path.getsize(f'{CLEAN}/{f}')/1e6:7.2f} MB")

order_head.csv          0.09 MB
order_lines.csv         1.13 MB
lanes.csv               0.24 MB
poc_reference.csv       0.05 MB
model_grids.npz         0.42 MB


## 5. The rules and the objective

From here on we build `dom_model.py` one piece at a time. Each piece gets a text cell explaining
the rule and a `%%writefile` cell holding the code that implements it.

`%%writefile` only writes — it does not run. Section 6 imports the finished file and checks it.

### 5.1 Header, settings and the data loader

`CFG` holds every tunable number, with the doc's values as defaults. The loader reads the four
artefacts saved above and rebuilds exactly the objects the cleaning code just had in memory.

In [16]:
%%writefile dom_model.py
"""
dom_model.py -- the DOM model in one place.

Written by 02_data_cleaning.ipynb. Notebooks 03 (baseline), 04 (greedy) and 05 (classical)
import this file, so all three run on the same data, the same constraints and the same
objective function. None of it is restated in those notebooks.

    Objective:  Revenue - Penalty - Shipping cost        (DOM Equations.docx, 2022)

    C1  one order goes to one DC only
    C2  you cannot fill more than what was ordered
    C3  stock must be there, and must still cover the next 5 days
    C4  move only if fill rate rises 5 points AND 100 cases
    C5  case picks + pallet picks must fit the DC's daily limit
    C6  one dock slot per order, docks are limited
    C7  penalty applies only if filled cases fall under the order's threshold
"""
import os, math
from collections import defaultdict
import numpy as np
import pandas as pd

# Every number we can tune lives here. The defaults are the values from the doc.
CFG = dict(
    MIN_FILL_LIFT_PP   = 0.05,   # C4: fill rate must go up 5 points
    MIN_CASE_LIFT      = 100.0,  # C4: and at least 100 more cases
    FORWARD_COVER_DAYS = 5,      # C3: stock must still last 5 more days
    LEAD_TIME_MILES    = 500.0,  # 1 transit day for every 500 miles
    DOCKS_PER_ORDER    = 1,      # C6: the doc says one dock per order
    SAFETY_STOCK_FRAC  = 0.0,    # hold back stock at the other DC (sensitivity only)
    REQUIRE_OBJ_GAIN   = True,   # greedy only: move an order only if the money improves
)

CLEAN = os.environ.get("DOM_CLEAN", "clean")     # notebooks set this before importing

# ---- order headers: one row per order --------------------------------------
head  = pd.read_csv(f"{CLEAN}/order_head.csv", parse_dates=["pgi", "rdd"])
HEAD  = head.set_index("Group_Flag").to_dict("index")        # dict = fast lookup
FOCUS = sorted(head.loc[head.is_focus,  "Group_Flag"])       # the 472 the model decides
CLEAN_ORDERS = sorted(head.loc[~head.is_focus, "Group_Flag"])  # the 637 that stay put

# ---- order lines: order -> [(sku, cases, price per case, cases per pallet)] --
_lines = pd.read_csv(f"{CLEAN}/order_lines.csv")
LINES = {}
for _gf, _g in _lines.groupby("Group_Flag", sort=False):
    LINES[_gf] = [(int(r.MaterialNumber), float(r.cases),
                   float(r.price_per_case), float(r.cpp))
                  for r in _g.itertuples(index=False)]

# ---- calendar, stock timelines, dock and pick grids ------------------------
_z    = np.load(f"{CLEAN}/model_grids.npz", allow_pickle=False)
DATES = pd.to_datetime(_z["dates"])
NT    = len(DATES)                                  # how many days
DIX   = {d: i for i, d in enumerate(DATES)}         # date -> position in the array
DCS   = [int(x) for x in _z["dcs"]]                 # the 8 DCs
DCIX  = {d: i for i, d in enumerate(DCS)}           # DC -> row number

POOL0 = {(int(a), int(b)): _z["pool_vals"][i].copy()          # (DC,SKU) -> stock per day
         for i, (a, b) in enumerate(_z["pool_keys"])}
SKU_AT_DC = defaultdict(set)                                  # which DC stocks which SKU
for (_d, _s) in POOL0:
    SKU_AT_DC[_d].add(_s)

DOCK0    = _z["dock"]                               # free dock slots per DC/day
DOCK_HAS = _z["dock_has"].astype(bool)              # True where we really have dock data
CP0, PP0 = _z["cp"], _z["pp"]                       # case / pallet pick room per DC/day

# ---- freight ---------------------------------------------------------------
_lane = pd.read_csv(f"{CLEAN}/lanes.csv")
SHIP = {(int(r.Plant), int(r.TargetZip)): float(r.Shipping_Cost)
        for r in _lane.itertuples(index=False)}
DIST = {(int(r.Plant), int(r.TargetZip)): float(r.Distance)
        for r in _lane.itertuples(index=False)}

Writing dom_model.py


### 5.2 C2 and C3 — how much can we fill

`avail` says how many cases we may take. With a window it returns the **smallest** value over the
next 5 days, which is the C3 rule: stock has to be there today *and* still cover the days after.

`take` and `give` move stock and also change every later day, so one order's decision is visible
to the next.

`evaluate` never fills more than was ordered — that is C2.

In [17]:
%%writefile -a dom_model.py


# ---------------------------------------------------------------- C2 and C3
def avail(P, d, s, t, window=0):
    a = P.get((d, s))                    # stock timeline for this DC + SKU
    if a is None:                        # this DC does not stock the SKU
        return 0.0
    hi = min(NT, t + window + 1)         # last day of the window
    if hi <= t:                          # window is empty
        return 0.0
    return max(0.0, a[t:hi].min())       # the tightest day decides


def take(P, d, s, t, q):
    a = P.get((d, s))
    if a is not None and q > 0:
        a[t:] -= q                       # remove from today and every later day


def give(P, d, s, t, q):
    a = P.get((d, s))
    if a is not None and q > 0:
        a[t:] += q                       # put it back the same way


def evaluate(P, gf, d, t, window=0, reserve=0.0):
    fills, by, tot, rev = [], {}, 0.0, 0.0
    for s, dem, price, cpp in LINES[gf]:                    # each SKU line of the order
        free = avail(P, d, s, t, window) * (1.0 - reserve)  # optional safety stock
        q    = min(dem, free)                               # C2: never more than ordered
        fills.append((s, q, price, cpp))                    # remember the fill
        by[s] = q                                           # per-SKU, used for the penalty
        tot  += q                                           # running total of cases
        rev  += q * price                                   # running total of money
    return fills, by, tot, rev

Appending to dom_model.py


### 5.3 C5 — case picks and pallet picks

Full pallets are pallet picks and the leftover is case picks. That is just integer division. The
two totals are then checked against the DC's room for that day.

In [18]:
%%writefile -a dom_model.py


# ---------------------------------------------------------------- C5
def picks(fills):
    cp = pp = 0.0
    for _, q, _, cpp in fills:
        if q <= 0:                     # nothing filled on this line
            continue
        full = math.floor(q / cpp)     # how many whole pallets
        pp  += full                    # those are pallet picks
        cp  += q - full * cpp          # the remainder are case picks
    return cp, pp

Appending to dom_model.py


### 5.4 C7 — penalty

If filled cases reach the order's threshold, no penalty. If not, the penalty is
missing cases × price × penalty rate.

`MinimumPenalty` and `MaximumPenalty` are **not** used. They are 0 on most rows, which means
"not set" rather than "a cap of zero", so using them would delete real penalties. They are also
absent from the doc's objective.

In [19]:
%%writefile -a dom_model.py


# ---------------------------------------------------------------- C7
def penalty_of(gf, filled_by_sku, total_filled):
    h = HEAD[gf]
    if h["ppc"] <= 0:                           # this customer has no penalty deal
        return 0.0
    if total_filled >= h["threshold_cases"]:    # we reached the threshold -> no penalty
        return 0.0
    p = 0.0
    for s, dem, price, _ in LINES[gf]:          # otherwise charge for what is missing
        p += (dem - filled_by_sku.get(s, 0.0)) * price * h["ppc"]
    return max(0.0, p)                          # never negative

Appending to dom_model.py


### 5.5 The objective

`Revenue − Penalty − Shipping`. Every notebook scores every solution through this one function,
so no two results can be measured on different rulers.

In [20]:
%%writefile -a dom_model.py


# ---------------------------------------------------------------- objective
def objective(rec):
    return rec["revenue"] - rec["pen"] - rec["ship"]   # money we keep

Appending to dom_model.py


### 5.6 Stage A — the default assignment (C1)

Put every order at its own DC and take the stock. Priority-8 orders go first because the doc says
to protect them, then the largest orders by revenue. One order, one DC — that is C1.

This one function does three jobs for the whole repo:

* it **is** Baseline 1 in notebook `03`,
* it gives the fill rate each order reaches at its own DC, which C4 measures against,
* it leaves `POOL_A`, the stock the greedy and the MILP start from.

In [21]:
%%writefile -a dom_model.py


# ---------------------------------------------------------------- C1, stage A
def stage_A():
    P = {k: v.copy() for k, v in POOL0.items()}      # work on a copy of the stock
    seq = head.sort_values(["prio", "revenue"],      # priority 8 first, then big money
                           ascending=[True, False])["Group_Flag"]
    out = {}
    for gf in seq:                                   # one order at a time
        h = HEAD[gf]
        d = h["default_dc"]                          # its own DC
        t = DIX[h["pgi"]]                            # its ship day
        fills, by, tot, rev = evaluate(P, gf, d, t, window=0)   # how much fits today
        for s, q, _, _ in fills:
            take(P, d, s, t, q)                      # remove that stock for good
        cp, pp = picks(fills)                        # pick workload it creates
        out[gf] = dict(dc=d, t=t, fills=fills, by=by, filled=tot, revenue=rev,
                       cp=cp, pp=pp,
                       pen=penalty_of(gf, by, tot),          # penalty if short
                       ship=SHIP.get((d, h["zipc"]), 0.0),   # freight from its own DC
                       cof=tot / h["ordered_cases"] if h["ordered_cases"] else 0.0)
    return P, out

Appending to dom_model.py


### 5.7 Divert checks — the new ship date and C4

A farther DC needs more transit time, so we ship no later than planned and no later than the due
date allows, then step back off Saturdays and Sundays. There is no holiday calendar in the pack,
so weekends are all we can exclude.

C4 is the gate that decides whether a move is worth making at all: it needs **both** a 5-point
lift in fill rate and at least 100 more cases.

> The doc writes the first test as `≥ 1.05 · OrderedQty`, which nothing can ever satisfy. The text
> beside it says *"at least 5% increase in the fill rate and increase of 100 cases"*, so we read
> the number as **0.05**. The POC's own 3 moves show lifts of 7.5–25% and 410–784 cases, which fits.

In [22]:
%%writefile -a dom_model.py


# ---------------------------------------------------------------- date + C4
def lead_time(d, z):
    dd = DIST.get((d, z))                    # miles on this DC -> zip lane
    if dd is None:                           # this lane does not exist
        return None
    return max(1, int(math.ceil(dd / CFG["LEAD_TIME_MILES"])))   # 1 day per 500 miles


def revised_pgi(gf, d):
    h  = HEAD[gf]
    lt = lead_time(d, h["zipc"])                          # transit days from this DC
    if lt is None:                                        # no lane to this customer
        return None, "no_lane"
    p = min(h["pgi"],                                     # not later than planned
            h["rdd"] - pd.Timedelta(days=lt))             # and still meets the due date
    while p.weekday() >= 5:                               # Saturday or Sunday
        p -= pd.Timedelta(days=1)                         # step back one day
    if p not in DIX:                                      # outside our calendar
        return None, "pgi_out_of_horizon"
    return p, None                                        # good ship date


def passes_gate(lift_cases, ordered_cases, cfg=None):
    cfg = cfg or CFG
    if lift_cases < cfg["MIN_FILL_LIFT_PP"] * ordered_cases:   # less than 5 points
        return "fail_5pct"
    if lift_cases < cfg["MIN_CASE_LIFT"]:                      # less than 100 cases
        return "fail_100cases"
    return None                                                # passed both

Appending to dom_model.py


### 5.8 The scorecard

One function, used by all three solvers and by the comparison notebook, so every row of the final
table is computed the same way. Everything is reported on the 472 focus orders; the objective is
also shown for the whole book so the 637 clean orders are never quietly dropped.

In [23]:
%%writefile -a dom_model.py


# ---------------------------------------------------------------- scorecard
def metrics(res, label, runtime=None):
    oc = sum(HEAD[g]["ordered_cases"] for g in FOCUS)     # cases ordered
    fl = sum(res[g]["filled"] for g in FOCUS)             # cases filled
    return dict(
        scenario        = label,
        objective_focus = sum(objective(res[g]) for g in FOCUS),   # money on focus orders
        objective_all   = sum(objective(res[g]) for g in res),     # money on all 1,109
        fill_rate       = fl / oc,
        cases_filled    = fl,
        orders_diverted = sum(1 for g in FOCUS if res[g]["diverted"]),
        penalty_cost    = sum(res[g]["pen"]  for g in FOCUS),
        shipping_cost   = sum(res[g]["ship"] for g in FOCUS),
        runtime_s       = runtime,
    )


def to_frame(res, label):
    """Per-order answer, in the one format 06_comparison reads."""
    return pd.DataFrame([
        dict(scenario=label, order=g, is_focus=g in set(FOCUS),
             default_dc=HEAD[g]["default_dc"], assigned_dc=res[g]["chosen_dc"],
             moved=bool(res[g]["diverted"]),
             ordered_cases=HEAD[g]["ordered_cases"], filled_cases=res[g]["filled"],
             cof=res[g]["cof"], revenue=res[g]["revenue"],
             penalty=res[g]["pen"], shipping=res[g]["ship"],
             objective=objective(res[g]))
        for g in sorted(res)])

Appending to dom_model.py


## 6. Check the model file loads and behaves

Copy `dom_model.py` next to the cleaned files, import it, and confirm it rebuilds the same funnel
and the same Stage-A numbers we just computed here. If this cell passes, notebooks `03`, `04` and
`05` are guaranteed to be working on identical inputs.

In [24]:
shutil.copy("dom_model.py", f"{WORK}/dom_model.py")     # live next to the cleaned files
sys_path_note = None

import sys, importlib
sys.path.insert(0, WORK)
os.environ["DOM_CLEAN"] = CLEAN                          # tell the module where to read
import dom_model
importlib.reload(dom_model)
from dom_model import *

print("orders:", len(HEAD), "| focus:", len(FOCUS), "| clean:", len(CLEAN_ORDERS))
assert len(FOCUS) == len(head[head.is_focus]), "focus list did not survive the round trip"

POOL_A, DEF = stage_A()
print("stage A objective (focus):", f"{sum(objective(DEF[g]) for g in FOCUS):,.0f}")
print("stage A fill rate (focus):",
      f"{sum(DEF[g]['filled'] for g in FOCUS)/sum(HEAD[g]['ordered_cases'] for g in FOCUS)*100:.2f}%")

orders: 1109 | focus: 472 | clean: 637
stage A objective (focus): 44,365,994
stage A fill rate (focus): 90.47%


### 6.1 Sanity check against the POC's own output

The POC file is Nestlé's own 2024 result. Ordered cases must match exactly. Fill rate should be
close but not identical, because their run reserved stock for forecast demand that is not in the
pack.

In [25]:
po = poc_ord.set_index("SalesDocument/GroupingIndicator")     # index by order id
V = pd.DataFrame([(DEF[g]["cof"],                             # our fill rate
                   float(po.loc[g,"Default_DC_COF"]),         # their fill rate
                   HEAD[g]["ordered_cases"],                  # our cases
                   float(po.loc[g,"OrderedQty_Cases"]))       # their cases
                  for g in DEF if g in po.index],
                 columns=["cof_ours","cof_poc","cases_ours","cases_poc"])

print("ordered cases match  :", round((V.cases_ours == V.cases_poc).mean(), 4))
print("fill rate identical  :", round((abs(V.cof_ours - V.cof_poc) < 1e-6).mean(), 4))
print("fill rate within 5pts:", round((abs(V.cof_ours - V.cof_poc) < 0.05).mean(), 4))
print("mean fill ours / POC :", round(V.cof_ours.mean(),4), "/", round(V.cof_poc.mean(),4))

ordered cases match  : 1.0
fill rate identical  : 0.6925
fill rate within 5pts: 0.8638
mean fill ours / POC : 0.895 / 0.9422


## 7. Assumptions

Every one of these exists because something is missing from the pack, not because it was
convenient. They apply to all three solvers equally, which is the point of putting them here.

| # | We assumed | Why |
|---|---|---|
| 1 | Focus = 447 + **25** = **472** | no throughput limit in the pack; full dock days used instead |
| 2 | Stock = `OpeningStock − Total_Reserved_Qty` | exact on 100% of rows |
| 3 | Lead time = `ceil(distance / 500)` | matches both POC lead-time columns at 100% |
| 4 | Pick limit = highest usage ever seen | the file gives usage, never a limit |
| 5 | Holidays = weekends only | no holiday file in the pack |
| 6 | "SKU in the forecast" = DC has stock rows for it | no forecast file in the pack |
| 7 | C4 read as **0.05**, not `1.05` | `1.05` is impossible; the text and the POC's 3 moves support 0.05 |
| 8 | Min/Max penalty not used | 0 means "not set", not "a cap of zero" |
| 9 | No dock check at 5083 and 5773 | no dock rows for them; zero would wrongly block all moves |
| 10 | No forecast reserve at other DCs | not in the pack; likely why the POC found only 3 moves |

## 8. What the other notebooks do with this

| notebook | imports | adds | writes |
|---|---|---|---|
| `03_baseline` | `dom_model` | nothing — Stage A **is** the baseline | `results_baseline.csv`, `metrics_baseline.csv` |
| `04_greedy` | `dom_model` | one greedy pass, two sort orders | `results_greedy_*.csv`, `metrics_greedy.csv`, reject log |
| `05_classical` | `dom_model` | the binary model, solved exactly | `results_classical.csv`, `metrics_classical.csv` |
| `06_comparison` | the result files only | nothing | the tables and charts |

None of them re-reads the raw pack, and none of them redefines a rule.